
# Three-candidate DA comparison with dual support scopes

This notebook compares already-trained hourly DA candidates under two explicit scopes:

- **FS3_FULL_SUPPORT**: compares only `FS3_XGB` vs `FS3_LEAR` on their full shared valid support.
- **THREE_WAY_OVERLAP**: compares `FS3_XGB`, `FS3_LEAR`, and `LEAR_STRICT` on strict three-way intersection support.

No retraining, retuning, feature regeneration, or prediction regeneration is performed.


In [ ]:

from __future__ import annotations

import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    from scipy.stats import spearmanr as _scipy_spearmanr
except Exception:
    _scipy_spearmanr = None

NOTEBOOK_START = time.perf_counter()
pd.set_option("display.max_columns", 220)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "AGENTS.md").exists():
    for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (parent / "AGENTS.md").exists():
            PROJECT_ROOT = parent
            break

OUTPUT_DIR = PROJECT_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da/candidate_comparison_new_metrics"
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

BUSINESS_TZ = "Europe/Amsterdam"
TARGET_LEADS = [0, 4]
K_VALUES = [1, 2, 4]
L_VALUES = [1, 2, 4]

MODEL_RUNS = {
    "FS3_XGB": {
        "label": "FS3 XGBoost",
        "run_dir": PROJECT_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da/runs/20260424_113834_xgboost_fs3_combo_promoted_benchmark",
        "preferred_model_name": "xgboost_fs3_combo_promoted",
    },
    "FS3_LEAR": {
        "label": "FS3 LEAR",
        "run_dir": PROJECT_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da/runs/20260425_124242_lear_fs3_combo_promoted_benchmark",
        "preferred_model_name": "lear_fs3_combo_promoted",
    },
    "LEAR_STRICT": {
        "label": "LEAR STRICT",
        "run_dir": PROJECT_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da/runs_lago_lear/20260508_122730_lago_lear_six_year_benchmark",
        "preferred_model_name": "lear_lago_direct_dplus4_strict_no_future_1092",
    },
}

SCOPE_DEFS = {
    "FS3_FULL_SUPPORT": ["FS3_XGB", "FS3_LEAR"],
    "THREE_WAY_OVERLAP": ["FS3_XGB", "FS3_LEAR", "LEAR_STRICT"],
}

for candidate_id, spec in MODEL_RUNS.items():
    if not Path(spec["run_dir"]).exists():
        raise FileNotFoundError(f"Missing run directory for {candidate_id}: {spec['run_dir']}")

display(Markdown("## Candidate Registry"))
display(
    pd.DataFrame(
        [
            {
                "candidate_id": cid,
                "candidate_label": spec["label"],
                "run_dir": str(spec["run_dir"]),
                "preferred_model_name": spec["preferred_model_name"],
            }
            for cid, spec in MODEL_RUNS.items()
        ]
    )
)

display(Markdown("## Scope Definitions"))
display(
    pd.DataFrame(
        [
            {"comparison_scope": scope, "candidate_ids": ", ".join(cids)}
            for scope, cids in SCOPE_DEFS.items()
        ]
    )
)


In [ ]:

def _to_bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    lowered = series.astype(str).str.strip().str.lower()
    true_values = {"1", "true", "t", "yes", "y"}
    false_values = {"0", "false", "f", "no", "n", "nan", "none", ""}
    out = pd.Series(index=series.index, dtype="boolean")
    out[lowered.isin(true_values)] = True
    out[lowered.isin(false_values)] = False
    return out.fillna(False).astype(bool)


def _pick_column(df: pd.DataFrame, candidates: list[str], *, required: bool = False, field: str = "") -> str | None:
    lower_map = {col.lower(): col for col in df.columns}
    for name in candidates:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    if required:
        raise KeyError(
            f"Could not map required field '{field}'. Tried {candidates}. Available columns: {list(df.columns)}"
        )
    return None


def load_one_candidate(candidate_id: str, spec: dict[str, object]) -> tuple[pd.DataFrame, dict[str, object]]:
    pred_path = Path(spec["run_dir"]) / "predictions_long.parquet"
    if not pred_path.exists():
        raise FileNotFoundError(
            f"Missing required prediction artifact for candidate '{candidate_id}'. Expected file: {pred_path}"
        )
    df = pd.read_parquet(pred_path)
    model_col = _pick_column(df, ["model", "model_id", "model_name"], required=False)
    preferred = str(spec.get("preferred_model_name") or "")
    if model_col is None:
        raise ValueError(f"No model column found for {candidate_id}: {pred_path}")
    unique_models = sorted(df[model_col].dropna().astype(str).unique().tolist())
    if preferred not in unique_models:
        raise ValueError(
            f"Candidate '{candidate_id}' expected model '{preferred}' not found. Available: {unique_models}"
        )
    df = df[df[model_col].astype(str) == preferred].copy()
    meta = {
        "candidate_id": candidate_id,
        "candidate_label": spec["label"],
        "selected_model": preferred,
        "pred_path": str(pred_path),
        "rows_loaded": int(len(df)),
    }
    return df, meta


def standardise_prediction_schema(df: pd.DataFrame, *, candidate_id: str, candidate_label: str) -> pd.DataFrame:
    frame = df.copy()
    model_col = _pick_column(frame, ["model_id", "model", "model_name"])
    origin_col = _pick_column(frame, ["forecast_origin_utc", "origin_timestamp", "origin_time", "forecast_origin", "origin"])
    delivery_col = _pick_column(
        frame,
        ["target_timestamp_utc", "delivery_start", "timestamp", "delivery_datetime", "target_timestamp", "datetime", "ds"],
        required=True,
        field="delivery_start",
    )
    lead_col = _pick_column(frame, ["lead_day", "lead", "lead_time", "lead_day_index"], required=True, field="lead_day")
    y_true_col = _pick_column(frame, ["y_true", "actual", "target", "price_actual", "y", "observed"], required=True, field="y_true")
    y_pred_col = _pick_column(frame, ["y_pred", "forecast", "prediction", "y_hat", "yhat", "pred"], required=True, field="y_pred")
    split_col = _pick_column(frame, ["dataset_split", "split"])

    out = pd.DataFrame(index=frame.index)
    out["candidate_id"] = candidate_id
    out["candidate_label"] = candidate_label
    out["model_id"] = frame[model_col].astype(str) if model_col is not None else pd.NA
    out["origin_timestamp"] = pd.to_datetime(frame[origin_col], utc=True, errors="coerce") if origin_col is not None else pd.NaT
    delivery_raw = pd.to_datetime(frame[delivery_col], errors="coerce", utc=False)
    if getattr(delivery_raw.dt, "tz", None) is None:
        out["delivery_start"] = pd.to_datetime(frame[delivery_col], utc=True, errors="coerce")
    else:
        out["delivery_start"] = delivery_raw.dt.tz_convert("UTC")
    out["lead_day"] = pd.to_numeric(frame[lead_col], errors="coerce").astype("Int64")
    out["y_true"] = pd.to_numeric(frame[y_true_col], errors="coerce")
    out["y_pred"] = pd.to_numeric(frame[y_pred_col], errors="coerce")
    out["split"] = frame[split_col].astype(str).str.lower().str.strip() if split_col is not None else pd.NA

    observed_true_cols = [
        "truth_observed",
        "is_observed",
        "observed",
        "target_observed",
        "target_valid",
        "valid_truth",
        "is_valid_target",
        "is_valid_truth",
        "is_actual",
        "is_observed_target",
    ]
    invalid_true_cols = [
        "is_interpolated",
        "interpolated",
        "synthetic",
        "is_synthetic",
        "target_synthetic",
        "truth_synthetic",
        "invalid_truth",
        "is_invalid_truth",
    ]
    for flag in observed_true_cols + invalid_true_cols:
        source = _pick_column(frame, [flag])
        if source is not None:
            out[f"flag__{flag}"] = frame[source]
    return out


def filter_valid_predictions(df_std: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, object]]:
    frame = df_std.copy()
    valid = pd.Series(True, index=frame.index)
    valid &= frame["y_true"].notna()
    valid &= frame["y_pred"].notna()
    valid &= frame["lead_day"].isin(TARGET_LEADS)
    if frame["split"].notna().any():
        valid &= frame["split"].astype(str).str.lower().eq("test")
    positive_flags = [
        "flag__truth_observed",
        "flag__is_observed",
        "flag__target_observed",
        "flag__target_valid",
        "flag__valid_truth",
        "flag__is_valid_target",
        "flag__is_valid_truth",
        "flag__is_actual",
        "flag__is_observed_target",
    ]
    negative_flags = [
        "flag__is_interpolated",
        "flag__interpolated",
        "flag__synthetic",
        "flag__is_synthetic",
        "flag__target_synthetic",
        "flag__truth_synthetic",
        "flag__invalid_truth",
        "flag__is_invalid_truth",
    ]
    used_pos, used_neg = [], []
    for col in positive_flags:
        if col in frame.columns:
            valid &= _to_bool_series(frame[col])
            used_pos.append(col.replace("flag__", ""))
    for col in negative_flags:
        if col in frame.columns:
            valid &= ~_to_bool_series(frame[col])
            used_neg.append(col.replace("flag__", ""))
    filtered = frame[valid].copy().reset_index(drop=True)
    dup_mask = filtered.duplicated(subset=["candidate_id", "lead_day", "delivery_start"], keep=False)
    if dup_mask.any():
        raise AssertionError("Duplicate rows after validity filtering on candidate/lead/timestamp.")
    return filtered, {
        "rows_before_filter": int(len(frame)),
        "rows_after_filter": int(len(filtered)),
        "used_positive_truth_flags": ", ".join(used_pos),
        "used_negative_truth_flags": ", ".join(used_neg),
    }


def restrict_to_common_support(valid_df: pd.DataFrame, candidate_ids: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    parts = []
    support_rows = []
    for lead in TARGET_LEADS:
        part = valid_df[valid_df["lead_day"] == lead].copy()
        sets = {
            cid: set(part.loc[part["candidate_id"] == cid, "delivery_start"].tolist())
            for cid in candidate_ids
        }
        common_ts = set.intersection(*sets.values()) if sets else set()
        common_ts = sorted(common_ts)
        keep = part[part["delivery_start"].isin(common_ts)].copy()
        parts.append(keep)
        per_ts = keep.groupby("delivery_start")["candidate_id"].nunique().reset_index(name="candidate_count")
        support_rows.append(
            {
                "lead_day": int(lead),
                "common_timestamps": int(len(common_ts)),
                "min_delivery_start": min(common_ts) if common_ts else pd.NaT,
                "max_delivery_start": max(common_ts) if common_ts else pd.NaT,
                "min_candidates_per_timestamp": int(per_ts["candidate_count"].min()) if not per_ts.empty else 0,
                "max_candidates_per_timestamp": int(per_ts["candidate_count"].max()) if not per_ts.empty else 0,
            }
        )
    common_df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=valid_df.columns)
    support_df = pd.DataFrame(support_rows).sort_values("lead_day").reset_index(drop=True)
    return common_df, support_df


def _daily_complete_subset(common_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    frame = common_df.copy()
    frame["delivery_start_local"] = frame["delivery_start"].dt.tz_convert(BUSINESS_TZ)
    frame["delivery_date"] = frame["delivery_start_local"].dt.date
    counts = (
        frame.groupby(["candidate_id", "candidate_label", "lead_day", "delivery_date"])["delivery_start"]
        .count()
        .rename("hours_in_day")
        .reset_index()
    )
    complete_keys = counts[counts["hours_in_day"] == 24][["candidate_id", "candidate_label", "lead_day", "delivery_date"]]
    daily = frame.merge(complete_keys, on=["candidate_id", "candidate_label", "lead_day", "delivery_date"], how="inner")
    check = daily.groupby(["candidate_id", "lead_day", "delivery_date"])["delivery_start"].count().reset_index(name="n_hours")
    assert check["n_hours"].eq(24).all(), "Daily operational metrics must use complete 24h days only."
    return daily, check


def _pct(series: pd.Series, q: float) -> float:
    if series.empty:
        return np.nan
    return float(np.nanpercentile(series.to_numpy(dtype=float), q))


def compute_conventional_metrics(df: pd.DataFrame, naive_mae_map: dict[int, float] | None = None) -> pd.DataFrame:
    rows = []
    for (cid, label, lead), part in df.groupby(["candidate_id", "candidate_label", "lead_day"]):
        err = part["y_pred"] - part["y_true"]
        ae = err.abs()
        row = {
            "candidate_id": cid,
            "candidate_label": label,
            "lead_day": int(lead),
            "MAE": float(ae.mean()),
            "RMSE": float(np.sqrt(np.mean(np.square(err)))),
            "bias": float(err.mean()),
            "median_AE": float(ae.median()),
            "p90_AE": _pct(ae, 90),
            "p95_AE": _pct(ae, 95),
            "n_rows": int(len(part)),
        }
        if naive_mae_map and int(lead) in naive_mae_map and naive_mae_map[int(lead)] > 0:
            row["rMAE"] = row["MAE"] / float(naive_mae_map[int(lead)])
        else:
            row["rMAE"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["lead_day", "candidate_id"]).reset_index(drop=True)


def compute_top_bottom_k_metrics(common_df: pd.DataFrame, k_values: list[int]) -> pd.DataFrame:
    daily, _ = _daily_complete_subset(common_df)
    rows = []
    for (cid, label, lead, ddate), part in daily.groupby(["candidate_id", "candidate_label", "lead_day", "delivery_date"]):
        day = part.sort_values("delivery_start")
        for k in k_values:
            act_top = set(day.sort_values(["y_true", "delivery_start"], ascending=[False, True]).head(k)["delivery_start"])
            pred_top = set(day.sort_values(["y_pred", "delivery_start"], ascending=[False, True]).head(k)["delivery_start"])
            act_bot = set(day.sort_values(["y_true", "delivery_start"], ascending=[True, True]).head(k)["delivery_start"])
            pred_bot = set(day.sort_values(["y_pred", "delivery_start"], ascending=[True, True]).head(k)["delivery_start"])
            rows.append(
                {
                    "candidate_id": cid,
                    "candidate_label": label,
                    "lead_day": int(lead),
                    "delivery_date": ddate,
                    "k": int(k),
                    "topk_high_hit_rate": len(act_top & pred_top) / float(k),
                    "bottomk_low_hit_rate": len(act_bot & pred_bot) / float(k),
                }
            )
    ddf = pd.DataFrame(rows)
    if ddf.empty:
        return pd.DataFrame(columns=["candidate_id", "candidate_label", "lead_day", "k"])
    out = (
        ddf.groupby(["candidate_id", "candidate_label", "lead_day", "k"])
        .agg(
            mean_topk_high_hit_rate=("topk_high_hit_rate", "mean"),
            median_topk_high_hit_rate=("topk_high_hit_rate", "median"),
            mean_bottomk_low_hit_rate=("bottomk_low_hit_rate", "mean"),
            median_bottomk_low_hit_rate=("bottomk_low_hit_rate", "median"),
            evaluated_complete_days=("delivery_date", "nunique"),
        )
        .reset_index()
        .sort_values(["lead_day", "k", "candidate_id"])
        .reset_index(drop=True)
    )
    return out


def _spearman_day(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if _scipy_spearmanr is not None:
        st = _scipy_spearmanr(y_true, y_pred).statistic
        return float(st) if st is not None else np.nan
    r1 = pd.Series(y_true).rank(method="average").to_numpy(dtype=float)
    r2 = pd.Series(y_pred).rank(method="average").to_numpy(dtype=float)
    if np.std(r1) == 0 or np.std(r2) == 0:
        return np.nan
    return float(np.corrcoef(r1, r2)[0, 1])


def compute_daily_spearman(common_df: pd.DataFrame) -> pd.DataFrame:
    daily, _ = _daily_complete_subset(common_df)
    rows = []
    for (cid, label, lead, ddate), part in daily.groupby(["candidate_id", "candidate_label", "lead_day", "delivery_date"]):
        day = part.sort_values("delivery_start")
        score = _spearman_day(day["y_true"].to_numpy(dtype=float), day["y_pred"].to_numpy(dtype=float))
        rows.append(
            {
                "candidate_id": cid,
                "candidate_label": label,
                "lead_day": int(lead),
                "delivery_date": ddate,
                "spearman": score,
            }
        )
    ddf = pd.DataFrame(rows)
    if ddf.empty:
        return pd.DataFrame(columns=["candidate_id", "candidate_label", "lead_day"])
    out = (
        ddf.groupby(["candidate_id", "candidate_label", "lead_day"])
        .agg(
            mean_spearman=("spearman", "mean"),
            median_spearman=("spearman", "median"),
            p10_spearman=("spearman", lambda s: np.nanpercentile(s, 10)),
            p90_spearman=("spearman", lambda s: np.nanpercentile(s, 90)),
            evaluated_complete_days=("delivery_date", "nunique"),
        )
        .reset_index()
        .sort_values(["lead_day", "candidate_id"])
        .reset_index(drop=True)
    )
    return out


def _rolling_avg(values: np.ndarray, L: int) -> np.ndarray:
    return np.convolve(values, np.ones(L, dtype=float) / float(L), mode="valid")


def compute_duration_window_regret(common_df: pd.DataFrame, L_values: list[int]) -> pd.DataFrame:
    daily, _ = _daily_complete_subset(common_df)
    rows = []
    for (cid, label, lead, ddate), part in daily.groupby(["candidate_id", "candidate_label", "lead_day", "delivery_date"]):
        day = part.sort_values("delivery_start")
        y_true = day["y_true"].to_numpy(dtype=float)
        y_pred = day["y_pred"].to_numpy(dtype=float)
        for L in L_values:
            true_roll = _rolling_avg(y_true, L)
            pred_roll = _rolling_avg(y_pred, L)
            idx_true_low = int(np.argmin(true_roll))
            idx_pred_low = int(np.argmin(pred_roll))
            idx_true_high = int(np.argmax(true_roll))
            idx_pred_high = int(np.argmax(pred_roll))
            rows.append(
                {
                    "candidate_id": cid,
                    "candidate_label": label,
                    "lead_day": int(lead),
                    "delivery_date": ddate,
                    "L": int(L),
                    "low_window_regret": float(true_roll[idx_pred_low] - true_roll[idx_true_low]),
                    "high_window_regret": float(true_roll[idx_true_high] - true_roll[idx_pred_high]),
                }
            )
    ddf = pd.DataFrame(rows)
    if ddf.empty:
        return pd.DataFrame(columns=["candidate_id", "candidate_label", "lead_day", "L"])
    out = (
        ddf.groupby(["candidate_id", "candidate_label", "lead_day", "L"])
        .agg(
            mean_low_window_regret=("low_window_regret", "mean"),
            median_low_window_regret=("low_window_regret", "median"),
            p90_low_window_regret=("low_window_regret", lambda s: np.nanpercentile(s, 90)),
            mean_high_window_regret=("high_window_regret", "mean"),
            median_high_window_regret=("high_window_regret", "median"),
            p90_high_window_regret=("high_window_regret", lambda s: np.nanpercentile(s, 90)),
            evaluated_complete_days=("delivery_date", "nunique"),
        )
        .reset_index()
        .sort_values(["lead_day", "L", "candidate_id"])
        .reset_index(drop=True)
    )
    return out


def build_summary_table(
    conventional: pd.DataFrame,
    top_bottom: pd.DataFrame,
    spearman: pd.DataFrame,
    duration_regret: pd.DataFrame,
) -> pd.DataFrame:
    summary = conventional.copy()
    if not top_bottom.empty:
        tb = top_bottom[["candidate_id", "candidate_label", "lead_day", "k", "mean_topk_high_hit_rate", "mean_bottomk_low_hit_rate"]].copy()
        top_piv = tb.pivot_table(index=["candidate_id", "candidate_label", "lead_day"], columns="k", values="mean_topk_high_hit_rate")
        bot_piv = tb.pivot_table(index=["candidate_id", "candidate_label", "lead_day"], columns="k", values="mean_bottomk_low_hit_rate")
        top_piv.columns = [f"top{int(k)}_high_hit_rate" for k in top_piv.columns]
        bot_piv.columns = [f"bottom{int(k)}_low_hit_rate" for k in bot_piv.columns]
        summary = summary.merge(top_piv.join(bot_piv, how="outer").reset_index(), on=["candidate_id", "candidate_label", "lead_day"], how="left")
    if not spearman.empty:
        summary = summary.merge(
            spearman[["candidate_id", "candidate_label", "lead_day", "mean_spearman", "median_spearman", "p10_spearman", "p90_spearman"]],
            on=["candidate_id", "candidate_label", "lead_day"],
            how="left",
        )
    if not duration_regret.empty:
        dr = duration_regret[["candidate_id", "candidate_label", "lead_day", "L", "mean_low_window_regret", "mean_high_window_regret"]].copy()
        low = dr.pivot_table(index=["candidate_id", "candidate_label", "lead_day"], columns="L", values="mean_low_window_regret")
        high = dr.pivot_table(index=["candidate_id", "candidate_label", "lead_day"], columns="L", values="mean_high_window_regret")
        low.columns = [f"mean_low_window_regret_L{int(v)}" for v in low.columns]
        high.columns = [f"mean_high_window_regret_L{int(v)}" for v in high.columns]
        summary = summary.merge(low.join(high, how="outer").reset_index(), on=["candidate_id", "candidate_label", "lead_day"], how="left")
    return summary.sort_values(["lead_day", "candidate_id"]).reset_index(drop=True)


def _load_official_naive_mae_for_scope(common_df: pd.DataFrame) -> tuple[dict[int, float] | None, str]:
    ref_run = Path(MODEL_RUNS["FS3_XGB"]["run_dir"])
    ref_json = ref_run / "official_naive_reference.json"
    if not ref_json.exists():
        return None, f"official_naive_reference.json missing at {ref_json}"
    ref = json.loads(ref_json.read_text(encoding="utf-8"))
    model = str(ref.get("model", "")).strip()
    if not model:
        return None, "official naive model missing in official_naive_reference.json"
    pred_path = ref_run / "predictions_long.parquet"
    if not pred_path.exists():
        return None, f"predictions file missing for naive source: {pred_path}"
    raw = pd.read_parquet(pred_path)
    std = standardise_prediction_schema(raw, candidate_id="OFFICIAL_NAIVE", candidate_label=f"Official {model}")
    if "model" in raw.columns:
        std["model_id"] = raw["model"].astype(str)
    std = std[std["model_id"].astype(str) == model].copy()
    if std.empty:
        return None, f"official naive model {model} not present in source predictions"
    valid_naive, _ = filter_valid_predictions(std)
    common_ts = common_df[["lead_day", "delivery_start"]].drop_duplicates()
    scoped = valid_naive.merge(common_ts, on=["lead_day", "delivery_start"], how="inner")
    out = {}
    for lead in TARGET_LEADS:
        p = scoped[scoped["lead_day"] == lead]
        if p.empty:
            return None, f"no naive rows for lead {lead} on this scope support"
        out[int(lead)] = float((p["y_true"] - p["y_pred"]).abs().mean())
    return out, f"official naive model {model} from {ref_run.name}"


def plot_conventional_metrics(conventional: pd.DataFrame, scope_name: str, save_path: Path) -> None:
    metrics = ["MAE", "RMSE", "bias", "p95_AE"]
    leads = sorted(conventional["lead_day"].astype(int).unique().tolist())
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()
    for ax, metric in zip(axes, metrics):
        for lead in leads:
            part = conventional[conventional["lead_day"] == lead].sort_values("candidate_label")
            x = np.arange(len(part))
            width = 0.38
            shift = -width / 2 if lead == min(leads) else width / 2
            ax.bar(x + shift, part[metric], width=width, label=f"lead_day={lead}")
            ax.set_xticks(x)
            ax.set_xticklabels(part["candidate_label"], rotation=20, ha="right")
        ax.set_title(f"{scope_name} | {metric}")
        ax.grid(axis="y", alpha=0.25)
    axes[0].legend(loc="best")
    fig.tight_layout()
    fig.savefig(save_path, dpi=140)
    plt.close(fig)



## Scope explanation

- **FS3_FULL_SUPPORT**: best scope for selecting between promoted FS3 candidates, because it uses their full shared valid timestamp support.
- **THREE_WAY_OVERLAP**: fair diagnostic scope including LEAR_STRICT, but not a full-period benchmark because support is strict three-way overlap.


In [ ]:

# 1) Load and validity-filter all candidates once
candidate_raw = {}
candidate_valid = {}
candidate_meta = {}
validity_audit_rows = []

for cid, spec in MODEL_RUNS.items():
    raw_df, load_meta = load_one_candidate(cid, spec)
    std_df = standardise_prediction_schema(raw_df, candidate_id=cid, candidate_label=spec["label"])
    std_df["model_id"] = load_meta["selected_model"]
    valid_df, filt_meta = filter_valid_predictions(std_df)

    candidate_raw[cid] = raw_df
    candidate_valid[cid] = valid_df
    candidate_meta[cid] = {**load_meta, **filt_meta}

    validity_audit_rows.append(
        {
            "candidate_id": cid,
            "candidate_label": spec["label"],
            "rows_before_filter": filt_meta["rows_before_filter"],
            "rows_after_filter": filt_meta["rows_after_filter"],
            "used_positive_truth_flags": filt_meta["used_positive_truth_flags"],
            "used_negative_truth_flags": filt_meta["used_negative_truth_flags"],
        }
    )

display(Markdown("## Candidate validity-filter counts"))
display(pd.DataFrame(validity_audit_rows).sort_values("candidate_id").reset_index(drop=True))

# 2) Build both scope datasets
scope_common = {}
scope_support = {}
support_rows = []

for scope_name, scope_candidates in SCOPE_DEFS.items():
    if scope_name == "FS3_FULL_SUPPORT":
        assert set(scope_candidates) == {"FS3_XGB", "FS3_LEAR"}, "FS3_FULL_SUPPORT must contain only FS3 candidates."
    if scope_name == "THREE_WAY_OVERLAP":
        assert set(scope_candidates) == {"FS3_XGB", "FS3_LEAR", "LEAR_STRICT"}, "THREE_WAY_OVERLAP must contain all three candidates."

    scope_valid_df = pd.concat([candidate_valid[cid] for cid in scope_candidates], ignore_index=True)
    common_df, support_df = restrict_to_common_support(scope_valid_df, scope_candidates)

    # Assertions for candidate count per retained timestamp
    expected_n = len(scope_candidates)
    ccheck = common_df.groupby(["lead_day", "delivery_start"])["candidate_id"].nunique().reset_index(name="candidate_count")
    assert ccheck["candidate_count"].eq(expected_n).all(), f"{scope_name}: retained timestamps do not have expected candidate count={expected_n}."
    assert set(common_df["lead_day"].dropna().astype(int).unique().tolist()) == set(TARGET_LEADS), f"{scope_name}: lead_day mismatch."
    assert common_df["y_true"].notna().all() and common_df["y_pred"].notna().all(), f"{scope_name}: null values entered metrics input."

    scope_common[scope_name] = common_df
    scope_support[scope_name] = support_df

    # per-candidate rows after support + complete days
    local = common_df.copy()
    local["delivery_local"] = local["delivery_start"].dt.tz_convert(BUSINESS_TZ)
    local["delivery_date"] = local["delivery_local"].dt.date
    day_counts = local.groupby(["candidate_id", "lead_day", "delivery_date"])["delivery_start"].count().reset_index(name="n_hours")
    comp_days = day_counts[day_counts["n_hours"] == 24].groupby(["candidate_id", "lead_day"])["delivery_date"].nunique().reset_index(name="complete_24h_days")
    rows_after = common_df.groupby(["candidate_id", "lead_day"])["delivery_start"].count().reset_index(name="rows_after_common_support")

    for _, row in support_df.iterrows():
        lead = int(row["lead_day"])
        support_rows.append(
            {
                "comparison_scope": scope_name,
                "scope_row_type": "scope_lead_summary",
                "candidate_id": "ALL",
                "lead_day": lead,
                "expected_candidate_count": expected_n,
                "common_timestamps": int(row["common_timestamps"]),
                "min_delivery_start": row["min_delivery_start"],
                "max_delivery_start": row["max_delivery_start"],
                "min_candidates_per_timestamp": int(row["min_candidates_per_timestamp"]),
                "max_candidates_per_timestamp": int(row["max_candidates_per_timestamp"]),
                "rows_after_common_support": np.nan,
                "complete_24h_days": np.nan,
            }
        )
        for cid in scope_candidates:
            r = rows_after[(rows_after["candidate_id"] == cid) & (rows_after["lead_day"] == lead)]
            c = comp_days[(comp_days["candidate_id"] == cid) & (comp_days["lead_day"] == lead)]
            support_rows.append(
                {
                    "comparison_scope": scope_name,
                    "scope_row_type": "scope_candidate_lead",
                    "candidate_id": cid,
                    "lead_day": lead,
                    "expected_candidate_count": expected_n,
                    "common_timestamps": int(row["common_timestamps"]),
                    "min_delivery_start": row["min_delivery_start"],
                    "max_delivery_start": row["max_delivery_start"],
                    "min_candidates_per_timestamp": int(row["min_candidates_per_timestamp"]),
                    "max_candidates_per_timestamp": int(row["max_candidates_per_timestamp"]),
                    "rows_after_common_support": int(r["rows_after_common_support"].iloc[0]) if not r.empty else 0,
                    "complete_24h_days": int(c["complete_24h_days"].iloc[0]) if not c.empty else 0,
                }
            )

support_by_scope = pd.DataFrame(support_rows).sort_values(["comparison_scope", "lead_day", "scope_row_type", "candidate_id"]).reset_index(drop=True)
display(Markdown("## Support by scope"))
display(support_by_scope)


In [ ]:

# 3) Compute metrics for each scope with same metric functions
all_conventional = []
all_top_bottom = []
all_spearman = []
all_duration = []
all_summary = []
scope_runtime_rows = []
saved_figures = []

for scope_name, common_df in scope_common.items():
    t0 = time.perf_counter()
    naive_map, naive_note = _load_official_naive_mae_for_scope(common_df)
    if naive_map is None:
        display(Markdown(f"**{scope_name} rMAE note:** skipped. Reason: {naive_note}"))
    else:
        display(Markdown(f"**{scope_name} rMAE note:** enabled ({naive_note})."))

    conventional = compute_conventional_metrics(common_df, naive_mae_map=naive_map)
    top_bottom = compute_top_bottom_k_metrics(common_df, k_values=K_VALUES)
    spearman = compute_daily_spearman(common_df)
    duration = compute_duration_window_regret(common_df, L_values=L_VALUES)
    summary = build_summary_table(conventional, top_bottom, spearman, duration)

    for df in [conventional, top_bottom, spearman, duration, summary]:
        df["comparison_scope"] = scope_name
        assert "comparison_scope" in df.columns

    all_conventional.append(conventional)
    all_top_bottom.append(top_bottom)
    all_spearman.append(spearman)
    all_duration.append(duration)
    all_summary.append(summary)

    # lightweight scope-specific figure
    fig_path = FIG_DIR / f"conventional_metrics_bar_{scope_name}.png"
    plot_conventional_metrics(conventional, scope_name=scope_name, save_path=fig_path)
    saved_figures.append(fig_path)

    scope_runtime_rows.append(
        {
            "comparison_scope": scope_name,
            "runtime_sec_metrics_and_plot": float(time.perf_counter() - t0),
        }
    )

conventional_all = pd.concat(all_conventional, ignore_index=True).reset_index(drop=True)
top_bottom_all = pd.concat(all_top_bottom, ignore_index=True).reset_index(drop=True)
spearman_all = pd.concat(all_spearman, ignore_index=True).reset_index(drop=True)
duration_all = pd.concat(all_duration, ignore_index=True).reset_index(drop=True)
summary_all = pd.concat(all_summary, ignore_index=True).reset_index(drop=True)

# Required checks: scope candidate sets
fs3_scope_cands = set(conventional_all[conventional_all["comparison_scope"] == "FS3_FULL_SUPPORT"]["candidate_id"].unique().tolist())
three_scope_cands = set(conventional_all[conventional_all["comparison_scope"] == "THREE_WAY_OVERLAP"]["candidate_id"].unique().tolist())
assert fs3_scope_cands == {"FS3_XGB", "FS3_LEAR"}, f"FS3_FULL_SUPPORT candidates mismatch: {fs3_scope_cands}"
assert three_scope_cands == {"FS3_XGB", "FS3_LEAR", "LEAR_STRICT"}, f"THREE_WAY_OVERLAP candidates mismatch: {three_scope_cands}"

# Save required outputs
p_conventional = OUTPUT_DIR / "candidate_comparison_conventional_metrics_by_scope.csv"
p_top_bottom = OUTPUT_DIR / "candidate_comparison_top_bottom_k_by_scope.csv"
p_spearman = OUTPUT_DIR / "candidate_comparison_spearman_by_scope.csv"
p_duration = OUTPUT_DIR / "candidate_comparison_duration_window_regret_by_scope.csv"
p_summary = OUTPUT_DIR / "candidate_comparison_summary_table_by_scope.csv"
p_support = OUTPUT_DIR / "candidate_comparison_support_by_scope.csv"

conventional_all.to_csv(p_conventional, index=False)
top_bottom_all.to_csv(p_top_bottom, index=False)
spearman_all.to_csv(p_spearman, index=False)
duration_all.to_csv(p_duration, index=False)
summary_all.to_csv(p_summary, index=False)
support_by_scope.to_csv(p_support, index=False)

display(Markdown("## FS3_FULL_SUPPORT summary"))
display(summary_all[summary_all["comparison_scope"] == "FS3_FULL_SUPPORT"].sort_values(["lead_day", "candidate_id"]).reset_index(drop=True))
display(Markdown("## THREE_WAY_OVERLAP summary"))
display(summary_all[summary_all["comparison_scope"] == "THREE_WAY_OVERLAP"].sort_values(["lead_day", "candidate_id"]).reset_index(drop=True))

display(Markdown("## Saved by-scope outputs"))
for p in [p_conventional, p_top_bottom, p_spearman, p_duration, p_summary, p_support]:
    print("-", p)
print("Figure directory:", FIG_DIR)
for fp in saved_figures:
    print("-", fp.name)

scope_runtime = pd.DataFrame(scope_runtime_rows)
display(Markdown("## Scope runtimes"))
display(scope_runtime)


In [ ]:

# Phase 3 plotting helpers
import matplotlib.dates as mdates


def _load_selected_weeks():
    run_root = PROJECT_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da/runs"
    week_runs = sorted([p for p in run_root.glob("*_case_week_selection") if p.is_dir()])
    if week_runs:
        latest = week_runs[-1]
        selected_path = latest / "selected_weeks.csv"
        if selected_path.exists():
            sw = pd.read_csv(selected_path)
            mapping = {
                "typical_winter": "winter_stress",
                "typical_summer": "summer_low",
                "high_volatility": "shoulder_normal",
            }
            weeks = {}
            for source_cat, target_name in mapping.items():
                part = sw[sw["category"].astype(str) == source_cat]
                if part.empty:
                    continue
                r = part.sort_values("candidate_rank").iloc[0]
                weeks[target_name] = (pd.Timestamp(str(r["week_start_local_date"])), pd.Timestamp(str(r["week_end_local_date"])))
            if len(weeks) == 3:
                return weeks, f"Objective weeks loaded from {selected_path}"
    fallback = {
        "winter_stress": (pd.Timestamp("2025-02-17"), pd.Timestamp("2025-02-23")),
        "summer_low": (pd.Timestamp("2025-08-18"), pd.Timestamp("2025-08-24")),
        "shoulder_normal": (pd.Timestamp("2024-12-09"), pd.Timestamp("2024-12-15")),
    }
    return fallback, "Fallback selected weeks used."


def _week_slice(scope_df: pd.DataFrame, lead_day: int, week_start_local: pd.Timestamp, week_end_local: pd.Timestamp) -> pd.DataFrame:
    part = scope_df[scope_df["lead_day"] == lead_day].copy()
    part["delivery_local"] = part["delivery_start"].dt.tz_convert(BUSINESS_TZ)
    local_date = part["delivery_local"].dt.date
    mask = (local_date >= week_start_local.date()) & (local_date <= week_end_local.date())
    return part.loc[mask].copy()


def plot_week_forecasts_by_scope(scope_name: str, week_name: str, week_start_local: pd.Timestamp, week_end_local: pd.Timestamp):
    scope_df = scope_common[scope_name]
    for lead in [0, 4]:
        week = _week_slice(scope_df, lead, week_start_local, week_end_local)
        if week.empty:
            continue
        fig, ax = plt.subplots(figsize=(12, 4.2))
        actual = week[["delivery_start", "y_true"]].drop_duplicates().sort_values("delivery_start")
        ax.plot(actual["delivery_start"], actual["y_true"], color="black", linewidth=2.2, label="Actual")
        for cid, part in week.groupby("candidate_id"):
            part = part.sort_values("delivery_start")
            label = part["candidate_label"].iloc[0]
            ax.plot(part["delivery_start"], part["y_pred"], linewidth=1.5, label=label)
        ax.set_title(f"{scope_name} | {week_name} | lead_day={lead}")
        ax.set_xlabel("Delivery time (UTC)")
        ax.set_ylabel("EUR/MWh")
        ax.grid(alpha=0.25)
        ax.legend(loc="upper right")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        fig.autofmt_xdate(rotation=20)
        # Avoid tight_layout warning with colorbar + multiple axes
        fig.subplots_adjust(left=0.08, right=0.92, top=0.86, bottom=0.16, wspace=0.28)
        out = FIG_DIR / f"week_forecast_{week_name}_D{lead}_{scope_name}.png"
        fig.savefig(out, dpi=140)
        plt.close(fig)
        saved_figures.append(out)


def plot_d0_d4_week_comparison_by_scope(scope_name: str, week_name: str, week_start_local: pd.Timestamp, week_end_local: pd.Timestamp):
    scope_df = scope_common[scope_name]
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)
    for row_i, lead in enumerate([0, 4]):
        week = _week_slice(scope_df, lead, week_start_local, week_end_local)
        if week.empty:
            axes[row_i].set_visible(False)
            continue
        actual = week[["delivery_start", "y_true"]].drop_duplicates().sort_values("delivery_start")
        axes[row_i].plot(actual["delivery_start"], actual["y_true"], color="black", linewidth=2.2, label="Actual")
        for cid, part in week.groupby("candidate_id"):
            part = part.sort_values("delivery_start")
            axes[row_i].plot(part["delivery_start"], part["y_pred"], linewidth=1.4, label=part["candidate_label"].iloc[0])
        axes[row_i].set_title(f"{scope_name} | {week_name} | {'D-only' if lead == 0 else 'D+4'}")
        axes[row_i].set_ylabel("EUR/MWh")
        axes[row_i].grid(alpha=0.25)
        axes[row_i].legend(loc="upper right")
        axes[row_i].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    axes[-1].set_xlabel("Delivery time (UTC)")
    fig.autofmt_xdate(rotation=20)
    # Avoid tight_layout warning with colorbar + multiple axes
    fig.subplots_adjust(left=0.08, right=0.92, top=0.86, bottom=0.16, wspace=0.28)
    out = FIG_DIR / f"week_d0_d4_{week_name}_{scope_name}.png"
    fig.savefig(out, dpi=140)
    plt.close(fig)
    saved_figures.append(out)


def plot_conventional_metrics_by_scope(scope_name: str):
    part = conventional_all[conventional_all["comparison_scope"] == scope_name].copy()
    metrics = ["MAE", "RMSE", "bias", "p95_AE"]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()
    leads = sorted(part["lead_day"].astype(int).unique().tolist())
    for ax, metric in zip(axes, metrics):
        for lead in leads:
            p = part[part["lead_day"] == lead].sort_values("candidate_label")
            x = np.arange(len(p))
            width = 0.38
            shift = -width / 2 if lead == min(leads) else width / 2
            ax.bar(x + shift, p[metric], width=width, label=f"lead_day={lead}")
            ax.set_xticks(x)
            ax.set_xticklabels(p["candidate_label"], rotation=20, ha="right")
        ax.set_title(f"{scope_name} | {metric} (lower is better)")
        ax.grid(axis="y", alpha=0.25)
    axes[0].legend(loc="best")
    # Avoid tight_layout warning with colorbar + multiple axes
    fig.subplots_adjust(left=0.08, right=0.92, top=0.86, bottom=0.16, wspace=0.28)
    out = FIG_DIR / f"conventional_metrics_bar_{scope_name}.png"
    fig.savefig(out, dpi=140)
    plt.close(fig)
    saved_figures.append(out)


def _plot_hit_heatmap(scope_name: str, metric_col: str, title_prefix: str, file_prefix: str):
    part = top_bottom_all[top_bottom_all["comparison_scope"] == scope_name].copy()
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    for ax_i, lead in enumerate([0, 4]):
        p = part[part["lead_day"] == lead]
        piv = p.pivot(index="candidate_label", columns="k", values=metric_col).sort_index()
        arr = piv.to_numpy(dtype=float)
        im = axes[ax_i].imshow(arr, aspect="auto", cmap="viridis", vmin=0.0, vmax=1.0)
        axes[ax_i].set_title(f"{scope_name} | lead_day={lead}")
        axes[ax_i].set_xticks(np.arange(len(piv.columns)))
        axes[ax_i].set_xticklabels([f"k={k}" for k in piv.columns])
        axes[ax_i].set_yticks(np.arange(len(piv.index)))
        axes[ax_i].set_yticklabels(piv.index)
        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                v = arr[i, j]
                axes[ax_i].text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if v < 0.6 else "black", fontsize=8)
    fig.suptitle(f"{title_prefix} (higher is better)", y=1.03)
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85)
    # Avoid tight_layout warning with colorbar + multiple axes
    fig.subplots_adjust(left=0.08, right=0.92, top=0.86, bottom=0.16, wspace=0.28)
    out = FIG_DIR / f"{file_prefix}_{scope_name}.png"
    fig.savefig(out, dpi=140, bbox_inches="tight")
    plt.close(fig)
    saved_figures.append(out)


def plot_hit_rate_heatmap_by_scope(scope_name: str):
    _plot_hit_heatmap(scope_name, "mean_topk_high_hit_rate", "Top-k high-price hit rate", "topk_high_hit_rate")
    _plot_hit_heatmap(scope_name, "mean_bottomk_low_hit_rate", "Bottom-k low-price hit rate", "bottomk_low_hit_rate")


def plot_spearman_by_scope(scope_name: str):
    part = spearman_all[spearman_all["comparison_scope"] == scope_name].copy()
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    order = sorted(part["candidate_label"].unique().tolist())
    leads = sorted(part["lead_day"].astype(int).unique().tolist())
    x = np.arange(len(order))
    width = 0.38
    for lead in leads:
        p = part[part["lead_day"] == lead].set_index("candidate_label").reindex(order).reset_index()
        y = p["mean_spearman"].to_numpy(dtype=float)
        low = p["p10_spearman"].to_numpy(dtype=float)
        high = p["p90_spearman"].to_numpy(dtype=float)
        yerr = np.vstack([y - low, high - y])
        shift = -width / 2 if lead == min(leads) else width / 2
        ax.bar(x + shift, y, width=width, yerr=yerr, capsize=4, label=f"lead_day={lead}")
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=20, ha="right")
    ax.set_ylabel("Spearman")
    ax.set_title(f"{scope_name} | Daily Spearman (higher is better)")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="best")
    # Avoid tight_layout warning with colorbar + multiple axes
    fig.subplots_adjust(left=0.08, right=0.92, top=0.86, bottom=0.16, wspace=0.28)
    out = FIG_DIR / f"spearman_by_candidate_{scope_name}.png"
    fig.savefig(out, dpi=140)
    plt.close(fig)
    saved_figures.append(out)


def _plot_duration(scope_name: str, value_col: str, ttl: str, filename: str):
    part = duration_all[duration_all["comparison_scope"] == scope_name].copy()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    leads = [0, 4]
    for ax_i, lead in enumerate(leads):
        p = part[part["lead_day"] == lead].copy()
        if p.empty:
            axes[ax_i].set_visible(False)
            continue
        order = sorted(p["candidate_label"].unique().tolist())
        x = np.arange(len(order))
        widths = 0.22
        for idx, L in enumerate([1, 2, 4]):
            q = p[p["L"] == L].set_index("candidate_label").reindex(order).reset_index()
            y = q[value_col].to_numpy(dtype=float)
            if "low" in value_col:
                p90_col = "p90_low_window_regret"
            else:
                p90_col = "p90_high_window_regret"
            p90 = q[p90_col].to_numpy(dtype=float)
            err = np.maximum(p90 - y, 0)
            shift = (idx - 1) * widths
            axes[ax_i].bar(x + shift, y, width=widths, yerr=err, capsize=3, label=f"L={L}")
        axes[ax_i].set_xticks(x)
        axes[ax_i].set_xticklabels(order, rotation=20, ha="right")
        axes[ax_i].set_title(f"{scope_name} | lead_day={lead}")
        axes[ax_i].grid(axis="y", alpha=0.25)
    axes[0].set_ylabel("Regret")
    axes[0].legend(loc="best")
    fig.suptitle(f"{ttl} (lower is better)", y=1.03)
    # Avoid tight_layout warning with colorbar + multiple axes
    fig.subplots_adjust(left=0.08, right=0.92, top=0.86, bottom=0.16, wspace=0.28)
    out = FIG_DIR / f"{filename}_{scope_name}.png"
    fig.savefig(out, dpi=140, bbox_inches="tight")
    plt.close(fig)
    saved_figures.append(out)


def plot_duration_regret_by_scope(scope_name: str):
    _plot_duration(scope_name, "mean_low_window_regret", "Duration low-window regret", "duration_regret_low")
    _plot_duration(scope_name, "mean_high_window_regret", "Duration high-window regret", "duration_regret_high")


def plot_support_coverage_by_scope():
    summary = support_by_scope[support_by_scope["scope_row_type"] == "scope_lead_summary"].copy()
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for i, metric in enumerate(["common_timestamps", "expected_candidate_count"]):
        piv = summary.pivot(index="comparison_scope", columns="lead_day", values=metric).reindex(["FS3_FULL_SUPPORT", "THREE_WAY_OVERLAP"])
        x = np.arange(len(piv.index))
        w = 0.35
        axes[i].bar(x - w / 2, piv[0].values, width=w, label="lead_day=0")
        axes[i].bar(x + w / 2, piv[4].values, width=w, label="lead_day=4")
        axes[i].set_xticks(x)
        axes[i].set_xticklabels(piv.index, rotation=15, ha="right")
        axes[i].set_title(metric.replace("_", " "))
        axes[i].grid(axis="y", alpha=0.25)
    axes[0].legend(loc="best")
    fig.suptitle("Support coverage by scope")
    # Avoid tight_layout warning with colorbar + multiple axes
    fig.subplots_adjust(left=0.08, right=0.92, top=0.86, bottom=0.16, wspace=0.28)
    out = FIG_DIR / "support_coverage_by_scope.png"
    fig.savefig(out, dpi=140)
    plt.close(fig)
    saved_figures.append(out)


In [ ]:

# Generate rich visualisations
selected_weeks, selected_weeks_note = _load_selected_weeks()
display(Markdown(f"### Selected weeks\n- {selected_weeks_note}"))
display(
    pd.DataFrame(
        [{"week_key": k, "start_local": v[0].date(), "end_local": v[1].date()} for k, v in selected_weeks.items()]
    )
)

# Week line plots per scope, D-only and D+4
for scope_name in SCOPE_DEFS.keys():
    for week_name, (wstart, wend) in selected_weeks.items():
        plot_week_forecasts_by_scope(scope_name, week_name, wstart, wend)

# Compact D0 vs D4 plots: generate for all selected weeks
for scope_name in SCOPE_DEFS.keys():
    for week_name, (wstart, wend) in selected_weeks.items():
        plot_d0_d4_week_comparison_by_scope(scope_name, week_name, wstart, wend)

# Metric plots by scope
for scope_name in SCOPE_DEFS.keys():
    plot_conventional_metrics_by_scope(scope_name)
    plot_hit_rate_heatmap_by_scope(scope_name)
    plot_spearman_by_scope(scope_name)
    plot_duration_regret_by_scope(scope_name)

# Optional support/coverage visual
plot_support_coverage_by_scope()

display(Markdown("### Figures generated"))
for p in sorted(set(saved_figures)):
    print("-", p.name)


In [ ]:

# Split interpretation block by scope
def _scope_pick(df: pd.DataFrame, scope: str) -> pd.DataFrame:
    return df[df["comparison_scope"] == scope].copy()


def _best_row(df: pd.DataFrame, lead: int, metric: str, lower=True):
    part = df[df["lead_day"] == lead].dropna(subset=[metric]).copy()
    if part.empty:
        return None
    idx = part[metric].idxmin() if lower else part[metric].idxmax()
    return part.loc[idx]


def _scope_interpret(scope: str):
    conv = _scope_pick(conventional_all, scope)
    sp = _scope_pick(spearman_all, scope)
    dr = _scope_pick(duration_all, scope)
    tb = _scope_pick(top_bottom_all, scope)

    lines = [f"### {scope} conclusion"]
    for lead in [0, 4]:
        mae_best = _best_row(conv, lead, "MAE", lower=True)
        rmse_best = _best_row(conv, lead, "RMSE", lower=True)
        sp_best = _best_row(sp, lead, "mean_spearman", lower=False)
        # operational proxy: mean total regret over L
        d = dr[dr["lead_day"] == lead].copy()
        if not d.empty:
            total = d.assign(total_regret=d["mean_low_window_regret"] + d["mean_high_window_regret"]).groupby(
                ["candidate_id", "candidate_label"], as_index=False
            )["total_regret"].mean()
            tbest = total.loc[total["total_regret"].idxmin()]
        else:
            tbest = None

        lines.append(f"- lead_day={lead}:")
        lines.append(
            f"  - Best MAE: {mae_best['candidate_label']} ({mae_best['MAE']:.3f})" if mae_best is not None else "  - Best MAE: n/a"
        )
        lines.append(
            f"  - Best RMSE: {rmse_best['candidate_label']} ({rmse_best['RMSE']:.3f})" if rmse_best is not None else "  - Best RMSE: n/a"
        )
        lines.append(
            f"  - Best mean Spearman: {sp_best['candidate_label']} ({sp_best['mean_spearman']:.3f})"
            if sp_best is not None
            else "  - Best mean Spearman: n/a"
        )
        lines.append(
            f"  - Lowest average duration regret proxy: {tbest['candidate_label']} ({tbest['total_regret']:.3f})"
            if tbest is not None
            else "  - Lowest average duration regret proxy: n/a"
        )

    display(Markdown("\n".join(lines)))


_scope_interpret("FS3_FULL_SUPPORT")
_scope_interpret("THREE_WAY_OVERLAP")

display(
    Markdown(
        "### Recommendation\n"
        "- Use **FS3_FULL_SUPPORT** for primary FS3 model selection.\n"
        "- Use **THREE_WAY_OVERLAP** as diagnostic evidence including LEAR_STRICT, not as a full-period benchmark.\n"
        "- If results are mixed across conventional and operational metrics, report that transparently rather than forcing one global winner."
    )
)



## Phase 3 — Rich visualisations and split interpretation

This phase keeps the metric engine unchanged and adds thesis-useful visual evidence for both scopes.

Visual interpretation guide:
- Conventional errors (MAE/RMSE/bias/p95_AE): **lower is better** (bias near zero preferred).
- Top/bottom-k hit rates and Spearman: **higher is better**.
- Duration-aware low/high window regret: **lower is better**.

Why duration-aware regret matters:
industrial operations often need contiguous 1h/2h/4h windows, so isolated correct cheap hours are not enough.


In [ ]:

# Final report block
support_counts = (
    support_by_scope[support_by_scope["scope_row_type"] == "scope_lead_summary"][
        ["comparison_scope", "lead_day", "common_timestamps", "min_delivery_start", "max_delivery_start"]
    ]
    .sort_values(["comparison_scope", "lead_day"])
    .reset_index(drop=True)
)
complete_counts = (
    support_by_scope[support_by_scope["scope_row_type"] == "scope_candidate_lead"][
        ["comparison_scope", "candidate_id", "lead_day", "complete_24h_days"]
    ]
    .sort_values(["comparison_scope", "lead_day", "candidate_id"])
    .reset_index(drop=True)
)

total_runtime = time.perf_counter() - NOTEBOOK_START
display(Markdown("## Common timestamp counts per lead-day and scope"))
display(support_counts)
display(Markdown("## Complete 24h local days per candidate/lead/scope"))
display(complete_counts)
display(Markdown(f"## Notebook runtime: {total_runtime:.2f} seconds"))
